# 4단계: 시각화
## 숫자를 그래프로 — 패턴을 눈으로 확인하기

---
### 이 실습에서 배우는 것
- matplotlib 기본 차트 (bar, line, pie)
- 이중 Y축 (매출 + 이익률 동시 표현)
- 서브플롯 구성
- 한글 폰트 설정

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# 한글 폰트 설정 (맥: AppleGothic, 윈도우: Malgun Gothic, 리눅스: NanumGothic)
import platform
if platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
elif platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
else:
    plt.rcParams['font.family'] = 'DejaVu Sans'  # 리눅스 기본

plt.rcParams['axes.unicode_minus'] = False  # 마이너스 부호 깨짐 방지
plt.rcParams['figure.dpi'] = 120

# 데이터 로드
df = pd.read_csv('data/coway_annual.csv')
df_q = pd.read_csv('data/coway_quarterly.csv')
df_os = pd.read_csv('data/coway_overseas.csv')

# 억원 단위
for col in ['매출액', '영업이익', '법인세전이익', '당기순이익', '국내매출', '해외매출', '기타매출']:
    df[col] = df[col] / 100
for col in ['매출액', '영업이익', '법인세전이익', '당기순이익']:
    df_q[col] = df_q[col] / 100

# 지표 계산
df['OPM'] = (df['영업이익'] / df['매출액'] * 100).round(1)
df['NPM'] = (df['당기순이익'] / df['매출액'] * 100).round(1)
df['해외비중'] = (df['해외매출'] / df['매출액'] * 100).round(1)
df_q['OPM'] = (df_q['영업이익'] / df_q['매출액'] * 100).round(1)

print('데이터 준비 완료')

## 4-1. 연간 매출 & 영업이익률 트렌드 (이중 Y축)

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))

# 왼쪽 Y축: 매출액 (막대)
bars = ax1.bar(df['연도'], df['매출액'], color='steelblue', alpha=0.7, label='매출액')
ax1.bar(df['연도'], df['영업이익'], color='orange', alpha=0.8, label='영업이익')
ax1.set_ylabel('금액 (억원)', fontsize=12)
ax1.set_ylim(0, df['매출액'].max() * 1.3)

# 막대 위에 숫자 표시
for i, (year, revenue) in enumerate(zip(df['연도'], df['매출액'])):
    ax1.text(year, revenue + 200, f'{revenue/10000:.2f}조', ha='center', fontsize=9)

# 오른쪽 Y축: OPM (선)
ax2 = ax1.twinx()
ax2.plot(df['연도'], df['OPM'], 'ro-', linewidth=2, markersize=8, label='OPM')
ax2.axhline(y=18, color='red', linestyle='--', alpha=0.4, linewidth=1)
ax2.set_ylabel('영업이익률 (%)', fontsize=12, color='red')
ax2.set_ylim(15, 22)
ax2.tick_params(axis='y', labelcolor='red')

# OPM 값 표시
for year, opm in zip(df['연도'], df['OPM']):
    ax2.annotate(f'{opm}%', (year, opm), textcoords='offset points',
                xytext=(0, 8), ha='center', fontsize=9, color='red')

# 범례 통합
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

ax1.set_title('코웨이 연간 매출액 & 영업이익률 추이', fontsize=14, fontweight='bold')
ax1.set_xticks(df['연도'])
plt.tight_layout()
plt.savefig('data/chart_annual_revenue.png', dpi=150, bbox_inches='tight')
plt.show()
print('차트 저장 완료')

## 4-2. 분기별 트렌드 — 계절성 & 이상 구간 포착

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

quarters = df_q['분기']
x = range(len(quarters))

# 차트 1: 분기별 매출 & 영업이익
axes[0].bar(x, df_q['매출액'], color='steelblue', alpha=0.6, label='매출액')
axes[0].bar(x, df_q['영업이익'], color='orange', alpha=0.8, label='영업이익')
axes[0].set_ylabel('금액 (억원)')
axes[0].set_title('분기별 매출액 & 영업이익', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(quarters, rotation=45, fontsize=8)
axes[0].legend()

# 연도 구분선 추가
for i in range(0, len(quarters), 4):
    axes[0].axvline(x=i-0.5, color='gray', linestyle='--', alpha=0.3)

# 차트 2: 분기별 OPM — 계절성 확인
colors = ['red' if opm < 15 else 'green' if opm > 18 else 'orange' for opm in df_q['OPM']]
axes[1].bar(x, df_q['OPM'], color=colors, alpha=0.7)
axes[1].axhline(y=18, color='green', linestyle='--', label='18% 기준선', alpha=0.6)
axes[1].axhline(y=15, color='red', linestyle='--', label='15% 경계선', alpha=0.6)
axes[1].set_ylabel('영업이익률 (%)')
axes[1].set_title('분기별 영업이익률 (OPM) — 초록: 18%↑ / 주황: 15-18% / 빨강: 15%↓', fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(quarters, rotation=45, fontsize=8)
axes[1].legend()

for i in range(0, len(quarters), 4):
    axes[1].axvline(x=i-0.5, color='gray', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig('data/chart_quarterly.png', dpi=150, bbox_inches='tight')
plt.show()

## 4-3. 사업부문 구성 — 파이차트 & 스택 바차트

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 2025년 사업부문 파이차트
latest = df[df['연도'] == 2025].iloc[0]
sizes = [latest['국내매출'], latest['해외매출'], latest['기타매출']]
labels = ['국내사업', '해외법인', '기타']
colors = ['#2196F3', '#FF9800', '#9E9E9E']
explode = (0.05, 0.05, 0)

axes[0].pie(sizes, labels=labels, colors=colors, explode=explode,
           autopct='%1.1f%%', startangle=90, pctdistance=0.85)
axes[0].set_title('2025년 사업부문별 매출 비중', fontweight='bold')

# 연도별 매출 구성 스택 바차트
국내 = df['국내매출'].values
해외 = df['해외매출'].values
기타 = df['기타매출'].values
years = df['연도'].values

axes[1].bar(years, 국내, label='국내사업', color='#2196F3', alpha=0.8)
axes[1].bar(years, 해외, bottom=국내, label='해외법인', color='#FF9800', alpha=0.8)
axes[1].bar(years, 기타, bottom=국내+해외, label='기타', color='#9E9E9E', alpha=0.8)

# 해외 비중 추이 표시
ax_right = axes[1].twinx()
ax_right.plot(years, df['해외비중'], 'k--o', linewidth=1.5, markersize=5, label='해외비중')
ax_right.set_ylabel('해외 비중 (%)')
ax_right.set_ylim(0, 60)

axes[1].set_title('연도별 사업부문 매출 구성 & 해외 비중', fontweight='bold')
axes[1].set_ylabel('매출액 (억원)')
axes[1].legend(loc='upper left')

plt.tight_layout()
plt.savefig('data/chart_segment.png', dpi=150, bbox_inches='tight')
plt.show()

## 4-4. 해외법인 비교 차트

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

법인들 = ['말레이시아', '미국', '태국']
colors_map = {'말레이시아': '#FF9800', '미국': '#2196F3', '태국': '#4CAF50'}

# 차트 1: 법인별 매출 추이
for 법인 in 법인들:
    data = df_os[df_os['법인'] == 법인]
    axes[0].plot(data['연도'], data['매출액_억원'], 'o-',
                color=colors_map[법인], label=법인, linewidth=2)
axes[0].set_title('해외법인별 매출 추이', fontweight='bold')
axes[0].set_ylabel('매출액 (억원)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 차트 2: 법인별 영업이익 (음수 포함)
for 법인 in 법인들:
    data = df_os[df_os['법인'] == 법인]
    axes[1].plot(data['연도'], data['영업이익_억원'], 'o-',
                color=colors_map[법인], label=법인, linewidth=2)
axes[1].axhline(y=0, color='black', linewidth=1, linestyle='--')
axes[1].fill_between(df_os[df_os['법인']=='미국']['연도'],
                      df_os[df_os['법인']=='미국']['영업이익_억원'],
                      0, alpha=0.15, color='red', label='미국 적자 구간')
axes[1].set_title('해외법인별 영업이익 (흑자전환 시점 주목)', fontweight='bold')
axes[1].set_ylabel('영업이익 (억원)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 차트 3: 2025년 법인별 렌탈 계정수
df_2025 = df_os[df_os['연도'] == 2025]
df_2025_main = df_2025[df_2025['법인'] != '기타해외']
bar_colors = [colors_map.get(l, 'gray') for l in df_2025_main['법인']]
axes[2].bar(df_2025_main['법인'], df_2025_main['렌탈계정수_천'],
           color=bar_colors, alpha=0.8)
axes[2].set_title('2025년 해외법인별 렌탈 계정수', fontweight='bold')
axes[2].set_ylabel('렌탈 계정수 (천 계정)')
for i, (법인, 계정) in enumerate(zip(df_2025_main['법인'], df_2025_main['렌탈계정수_천'])):
    axes[2].text(i, 계정 + 30, f'{계정:,}천', ha='center', fontsize=10)
axes[2].grid(True, alpha=0.3, axis='y')

plt.suptitle('코웨이 해외법인 분석', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('data/chart_overseas.png', dpi=150, bbox_inches='tight')
plt.show()

## 4-5. 렌탈 계정수 성장 & ARPU

In [ ]:
df['ARPU_만원'] = (df['매출액'] * 10000 / (df['렌탈계정수_천'] * 10)).round(1)

fig, ax1 = plt.subplots(figsize=(10, 5))

# 렌탈 계정수 (막대)
ax1.bar(df['연도'], df['렌탈계정수_천'] / 1000, color='#4CAF50', alpha=0.6, label='렌탈 계정수')
ax1.set_ylabel('렌탈 계정수 (백만 계정)', color='#2E7D32')
ax1.set_ylim(0, 15)

# 계정수 표시
for year, cnt in zip(df['연도'], df['렌탈계정수_천']):
    ax1.text(year, cnt/1000 + 0.1, f'{cnt/100:.0f}만', ha='center', fontsize=9)

# ARPU (오른쪽 Y축)
ax2 = ax1.twinx()
ax2.plot(df['연도'], df['ARPU_만원'], 'bs-', linewidth=2, markersize=8, label='ARPU')
ax2.set_ylabel('ARPU (만원/계정/년)', color='blue')
ax2.tick_params(axis='y', labelcolor='blue')

for year, arpu in zip(df['연도'], df['ARPU_만원']):
    ax2.annotate(f'{arpu:.0f}만', (year, arpu), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=9, color='blue')

ax1.set_title('렌탈 계정수 & ARPU 추이\n(계정수↑ + ARPU↑ = 매출 성장의 이상적 조합)', fontweight='bold')
ax1.set_xticks(df['연도'])

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.tight_layout()
plt.savefig('data/chart_arpu.png', dpi=150, bbox_inches='tight')
plt.show()

---
## ✏️ 과제

1. 4Q만 골라서 연도별 4Q OPM을 막대그래프로 그려보세요. 4Q가 항상 낮은가요?
2. 말레이시아 매출을 MYR(링깃) 기준과 KRW(원화) 기준으로 각각 선그래프를 그려보세요.
   - 힌트: `매출액_억원`이 이미 원화 환산값, `매출액_억원 / 환율_연평균 × 100`이 현지통화값
3. 해외 비중이 40%를 넘는 시점을 예측해보세요. (현재 추세 연장 가정)